In [0]:
import sys
from pathlib import Path

# Find the repository root from the notebook's directory
current = Path.cwd()

repo_root = next(
    (
        path
        for path in [current, *current.parents]
        if (path / "src" / "house_price_simple_analysis").is_dir()
    ),
    None
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate src/house_price_simple_analysis. "
        f"Current directory: {current}"
    )

sys.path.insert(0, str(repo_root / "src"))

print("Repository root:", repo_root)
print("Python source directory:", repo_root / "src")

In [0]:
import importlib

import house_price_simple_analysis.config as config_module
import house_price_simple_analysis.regression_model as regression_module

# Reload configuration first
importlib.reload(config_module)

# Then reload regression model
importlib.reload(regression_module)

ProjectConfig = config_module.ProjectConfig
RegressionModel = regression_module.RegressionModel

In [0]:
config = ProjectConfig.from_yaml(
    str(repo_root / "config" / "project_config.yml")
)

regression = RegressionModel(
    config=config,
    spark=spark
)


print(config.model.registered_model_name)

### Testing function Load Data

In [0]:
# 1. Load data
regression.load_data()

print("Training dataset:", regression.train_set.shape)
print("Testing dataset:", regression.test_set.shape)

In [0]:
regression.train_set.head()

In [0]:
regression.test_set.head()

### Prepare the model

In [0]:
# 2. Prepare model
regression.prepare_model()

### Training the model

In [0]:
# 3. Train model
regression.train()

### Evaluate the model

In [0]:
# 4. Evaluate model
metrics = regression.evaluate()

print("Evaluation metrics:", metrics)

### logging model using mlflow

In [0]:
# 5. Log to MLflow
model_uri = regression.log_model()

print("MLflow model URI:", model_uri)

### Testing register model

You can choose the model you want to register by manually adding the model's URL; if you don't, the function will automatically add the URL generated after the last time you logged the model.

In [0]:
regression.model_uri = "models:/m-c68f3c7d426b4f619809a2da79c55888"

In [0]:
# Register the model
registered_uri = regression.register_model()

print("Registered model:", regression.registered_model_name)
print("Version:", regression.registered_model_version)
print("URI:", registered_uri)

### Retrieving the registed model

In [0]:
import mlflow
import pandas as pd

mlflow.set_registry_uri("databricks-uc")

loaded_model = mlflow.sklearn.load_model(registered_uri)

sample = pd.DataFrame({
    "square_footage": [2000.0]
})

prediction = loaded_model.predict(sample)

print("Predicted price:", prediction[0])

### Made an Alias

At this time of the notebook, If the notebook get started you don't need to load data, train and register the model again. You only need to upload the existing registed model to a new instants and then use it.

If you have your model iniciated, you can use the build-in function to assing a alias.

```python
regression.set_model_alias("Champion")
```

Then use the alias:

```python
import mlflow

mlflow.set_registry_uri("databricks-uc")

model_uri = "models:/mlops_dev.house_pricing.house_price_model@Champion"

model = mlflow.sklearn.load_model(model_uri)

sample = pd.DataFrame({
    "square_footage": [2000.0]
})

print(model.predict(sample))
```


In this notebook, as it has restarted, we need to do the registration of the alias munually. It is done using a mlflow client.

In [0]:
## retriving an existing model

import mlflow
import mlflow.sklearn
import pandas as pd

mlflow.set_registry_uri("databricks-uc")

model_name = "mlops_dev.house_pricing.house_price_model"

# Load your existing registered model (version 1 in this case)
model_uri = f"models:/{model_name}/1"

model = mlflow.sklearn.load_model(model_uri)

In [0]:
# you can immediately generate predictions
sample = pd.DataFrame({
    "square_footage": [2000.0]
})

prediction = model.predict(sample)

print("Predicted house price:", prediction[0])

In [0]:
# Assign the Champion alias to your existing model

from mlflow import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name=model_name,
    alias="Champion",
    version="1"
)

In [0]:
# Now, instead of referencing version 1 directly, you can load the model through its alias

model_uri = f"models:/{model_name}@Champion"

model = mlflow.sklearn.load_model(model_uri)

### Testing endpoint

we implemented an the model using the databricks UI. We can testing it using the next code.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

response = w.serving_endpoints.query(
    name="house-price-endpoint",
    dataframe_records=[
        {"square_footage": 2000.0}
    ]
)

print(response.predictions)